In [0]:
from pyspark.sql import functions as F

silver_risk_df = spark.table("fraud_detection.silver.transactions_risk_flagged")
product_catalog_df = spark.table("fraud_detection.bronze.product_catalog_raw")

# collapse the catalog to one row per category: overall min/max across its products
category_price_ranges = (
    product_catalog_df
    .groupBy("product_category")
    .agg(
        F.min("typical_min_price").alias("category_min_price"),
        F.max("typical_max_price").alias("category_max_price")
    )
)

display(category_price_ranges)

In [0]:
silver_with_price_check = silver_risk_df.join(
    category_price_ranges,
    on="product_category",
    how="left"
).withColumn(
    "price_out_of_range",
    (F.col("transaction_amount") < F.col("category_min_price")) |
    (F.col("transaction_amount") > F.col("category_max_price"))
)

print(f"Row count: {silver_with_price_check.count()}")

display(silver_with_price_check.groupBy("price_out_of_range").count())

In [0]:
display(
    silver_with_price_check
    .groupBy("price_out_of_range")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct")
    )
)

In [0]:
display(spark.sql("DESCRIBE fraud_detection.bronze.product_catalog_raw"))

In [0]:
category_price_ranges = (
    product_catalog_df
    .withColumn("typical_min_price", F.col("typical_min_price").cast("double"))
    .withColumn("typical_max_price", F.col("typical_max_price").cast("double"))
    .groupBy("product_category")
    .agg(
        F.min("typical_min_price").alias("category_min_price"),
        F.max("typical_max_price").alias("category_max_price")
    )
)

display(category_price_ranges)

In [0]:
silver_with_price_check = silver_risk_df.join(
    category_price_ranges,
    on="product_category",
    how="left"
).withColumn(
    "price_out_of_range",
    (F.col("transaction_amount") < F.col("category_min_price")) |
    (F.col("transaction_amount") > F.col("category_max_price"))
)

display(silver_with_price_check.groupBy("price_out_of_range").count())

In [0]:
display(
    silver_with_price_check
    .groupBy("price_out_of_range")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct")
    )
)

In [0]:
final_price_columns = [c for c in silver_with_price_check.columns]

silver_with_price_check.write.mode("overwrite").saveAsTable("fraud_detection.silver.transactions_price_checked")

display(spark.sql("SELECT COUNT(*) FROM fraud_detection.silver.transactions_price_checked"))